# Results Analysis

Visualizations for the cross-domain HAR experiments.

- **Phase 1**: 7 LLM backbones with LN-only fine-tuning
- **Phase 3**: 4-level ablation across 4 LLMs and 4 datasets

Set `RESULTS_DIR` to the folder containing the two CSV files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

RESULTS_DIR = '../results'
FIG_DIR = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Phase 3 ablation results
df = pd.read_csv(os.path.join(RESULTS_DIR, 'phase3_ablation_results.csv'))
xd  = df[df['source'] != df['target']].copy()
id_ = df[df['source'] == df['target']].copy()

LEVEL_ORDER   = ['L1-Fixes', 'L2a-Reprogram', 'L2b-TwoStage', 'L3-Full']
LLM_ORDER     = ['GPT-2', 'Qwen-0.5B', 'Llama-1B', 'Qwen-1.5B']
DATASET_ORDER = ['uci', 'shoaib', 'motionsense', 'hhar']
DATASET_LABELS = {'uci': 'UCI HAR', 'shoaib': 'Shoaib',
                  'motionsense': 'MotionSense', 'hhar': 'HHAR'}

LLM4HAR_BASELINE = 71.42   # LLM4HAR paper XD macro F1 (Jin et al., KDD 2025)

COLORS = {
    'GPT-2':     '#4C72B0',
    'Qwen-0.5B': '#DD8452',
    'Llama-1B':  '#55A868',
    'Qwen-1.5B': '#C44E52',
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})
print('Setup complete. Figures will save to:', FIG_DIR)

## Figure 1: Cross-Domain F1 by Ablation Level and Backbone

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(LEVEL_ORDER))
n_llms  = len(LLM_ORDER)
width   = 0.18
offsets = np.linspace(-(n_llms-1)/2, (n_llms-1)/2, n_llms) * width

for i, llm in enumerate(LLM_ORDER):
    vals = []
    for level in LEVEL_ORDER:
        sub = xd[(xd['level'] == level) & (xd['llm'] == llm)]
        vals.append(sub['f1_macro'].mean() * 100 if len(sub) > 0 else 0)
    bars = ax.bar(x + offsets[i], vals, width,
                  label=llm, color=COLORS[llm], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, rotation=90)

ax.axhline(LLM4HAR_BASELINE, color='black', linestyle='--',
           linewidth=1.4, label=f'LLM4HAR baseline ({LLM4HAR_BASELINE}%)')

ax.set_xticks(x)
ax.set_xticklabels(LEVEL_ORDER, fontsize=11)
ax.set_ylabel('Cross-Domain Macro F1 (%)')
ax.set_title('Cross-Domain F1 by Ablation Level and Backbone')
ax.set_ylim(60, 80)
ax.legend(loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig1_ablation_xd_f1.png'), bbox_inches='tight')
plt.show()
print('Saved: fig1_ablation_xd_f1.png')

## Figure 2: Transfer Matrix — Best Config (Qwen-1.5B × L2b-TwoStage)

In [ ]:
best_llm   = 'Qwen-1.5B'
best_level = 'L2b-TwoStage'

sub = df[(df['llm'] == best_llm) & (df['level'] == best_level)].copy()

matrix = pd.DataFrame(index=DATASET_ORDER, columns=DATASET_ORDER, dtype=float)
for _, row in sub.iterrows():
    matrix.loc[row['source'], row['target']] = row['f1_macro'] * 100

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(matrix.values.astype(float), cmap='YlGn', vmin=55, vmax=95)

ax.set_xticks(range(len(DATASET_ORDER)))
ax.set_yticks(range(len(DATASET_ORDER)))
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASET_ORDER], fontsize=11)
ax.set_yticklabels([DATASET_LABELS[d] for d in DATASET_ORDER], fontsize=11)
ax.set_xlabel('Target Dataset')
ax.set_ylabel('Source Dataset')
ax.set_title(f'Macro F1 (%) — {best_llm} × {best_level}')

for i in range(len(DATASET_ORDER)):
    for j in range(len(DATASET_ORDER)):
        val = matrix.values[i, j]
        tag = ' (ID)' if i == j else ''
        color = 'white' if float(val) > 78 else 'black'
        ax.text(j, i, f'{float(val):.1f}{tag}', ha='center', va='center',
                fontsize=10, color=color, fontweight='bold' if i == j else 'normal')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Macro F1 (%)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig2_heatmap_best_config.png'), bbox_inches='tight')
plt.show()
print('Saved: fig2_heatmap_best_config.png')

## Figure 3: Improvement over Phase 1 (LN-Only) Baseline

In [ ]:
# Phase 1 cross-domain F1 for the 4 LLMs that appear in the ablation
PHASE1_BASELINES = {
    'GPT-2':     71.02,   # our Phase 1 run
    'Qwen-0.5B': 68.54,
    'Llama-1B':  71.73,
    'Qwen-1.5B': 72.50,
}

fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(LLM_ORDER))
width   = 0.19
offsets = np.linspace(-(len(LEVEL_ORDER)-1)/2,
                       (len(LEVEL_ORDER)-1)/2, len(LEVEL_ORDER)) * width

level_colors = ['#7fc97f', '#beaed4', '#fdc086', '#386cb0']

for i, level in enumerate(LEVEL_ORDER):
    deltas = []
    for llm in LLM_ORDER:
        sub  = xd[(xd['level'] == level) & (xd['llm'] == llm)]
        this = sub['f1_macro'].mean() * 100 if len(sub) > 0 else 0
        deltas.append(this - PHASE1_BASELINES[llm])
    bars = ax.bar(x + offsets[i], deltas, width, label=level,
                  color=level_colors[i], edgecolor='white', linewidth=0.5)
    for bar, d in zip(bars, deltas):
        ypos = bar.get_height() + 0.1 if d >= 0 else bar.get_height() - 0.4
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{d:+.1f}', ha='center', va='bottom', fontsize=8)

ax.axhline(0, color='black', linewidth=1.0, linestyle='-')
ax.set_xticks(x)
ax.set_xticklabels(LLM_ORDER, fontsize=11)
ax.set_ylabel('\u0394Macro F1 vs. Phase 1 Baseline (%)')
ax.set_title('Improvement over Phase 1 LN-Only Baseline per Ablation Level')
ax.legend(loc='upper left', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig3_delta_vs_phase1.png'), bbox_inches='tight')
plt.show()
print('Saved: fig3_delta_vs_phase1.png')

## Figure 4: Transfer Target Difficulty (Qwen-1.5B × L2b-TwoStage)

In [ ]:
sub = xd[(xd['llm'] == 'Qwen-1.5B') & (xd['level'] == 'L2b-TwoStage')].copy()

target_means = (sub.groupby('target')['f1_macro']
                   .mean()
                   .reindex(DATASET_ORDER) * 100)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.barh([DATASET_LABELS[d] for d in DATASET_ORDER],
               target_means.values,
               color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
               edgecolor='white')

for bar, val in zip(bars, target_means.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.axvline(LLM4HAR_BASELINE, color='black', linestyle='--',
           linewidth=1.3, label=f'LLM4HAR baseline ({LLM4HAR_BASELINE}%)')
ax.set_xlabel('Avg Macro F1 as Transfer Target (%)')
ax.set_title('Transfer Target Difficulty\n(Qwen-1.5B \u00d7 L2b-TwoStage)')
ax.set_xlim(50, 95)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig4_target_difficulty.png'), bbox_inches='tight')
plt.show()
print('Saved: fig4_target_difficulty.png')

## Phase 1 Summary Table

In [ ]:
p1 = pd.read_csv(os.path.join(RESULTS_DIR, 'phase1_multi_llm_results.csv'))
p1_xd = p1[p1['source'] != p1['target']]

summary = (p1_xd.groupby('llm')
              .agg(XD_Acc=('accuracy', lambda x: x.mean() * 100),
                   XD_F1 =('f1_macro',  lambda x: x.mean() * 100))
              .round(2)
              .sort_values('XD_F1', ascending=False))

summary['Delta_F1_vs_paper'] = (summary['XD_F1'] - 71.42).round(2)
print('Cross-domain averages (Phase 1, 7 LLMs):')
print(summary.to_string())
print('\nPaper baseline (GPT-2, LLM4HAR KDD 2025): Acc=79.86%, F1=71.42%')